In [3]:
import numpy as np
import scipy as sp
import json
import glob
from scipy.io import mmread
import io

In [4]:
algorithms_mqlib = [
    'AlgoAugmentedLagrangianPenExactType2MQLIB',
#    'AlgoAugmentedLagrangianPenExactTypeXMQLIB',
]


time_limits = [1, 2, 4, 10]
time_limits = [4]
sizes = [40, 80, 100, 120, 140, 160]


file_base_mqlib ='results/{}/{}/'
mqlib_input_data_mask = 'input_data/*.mqlib'
mqlib_input_meta_data_mask = 'input_data/meta/'
mqlib_result = '*.mqlib.output_time_limit_{}'


def get_data_mqlib(file_base, input_data_mask, input_meta_data_mask, algos, sizes, time_limits):
    for algo in algos:
        for size in sizes:
            input_data_files_mask = file_base.format(algo, size) + input_data_mask
            files = glob.glob(input_data_files_mask)

            for file in files:
                print(f'Reading {file}')
                with open(file) as f:
                    first_line = f.readline()
                    n, num_entries = first_line.split(' ')
                    entries = f.read()                
                f = io.StringIO(
                    "%%MatrixMarket matrix coordinate real symmetric\n" 
                    + f'{n} {n} {num_entries}\n'
                    + entries
                )
                Q = -mmread(f)

                file_meta = file.replace('input_data', 'input_data/meta').replace('mqlib', 'json')
                with open(file_meta) as f:
                    meta_data = json.load(f)

                for time_limit in time_limits:
                    file_mqlib_result = file_meta.replace('input_data/meta', '').replace('json', f'mqlib.output_time_limit_{time_limit}')

                    try:
                        with open(file_mqlib_result) as f:
                            result_data = f.readlines()
                    except:
                        print(f'No result data for {file_mqlib_result}. Skipping.')
                        continue

                    x = np.asarray(result_data[-1].split(' '), dtype=int)

                    del meta_data['optimum']
                
                    yield {'size': size, 'Q': Q, 'initial_estimate': x, **meta_data, 'additional_info': {'initial_astimate_algo': 'mqlib', 'time_limit': time_limit}}

        

In [5]:
data = get_data_mqlib(file_base_mqlib, mqlib_input_data_mask, mqlib_input_meta_data_mask, algorithms_mqlib, sizes, time_limits)    

In [11]:
data_list = list(data)

In [12]:
def to_biqbin_repr(data):
    q = data['Q']
    multiplication = 1
    # if not divisible by 2, double it so Biqbin can take the input
    if not np.all(q.data % 2 == 0):
        multiplication = 2
        q.data = q.data * 2
        data["offset"] *= 2
    else:
        print('all are even')
    
    return {'qubo': {
        'shape': q.shape,
        'nnz': q.nnz,
        'row': q.row.tolist(),
        'col': q.col.tolist(),
        'data': q.data.tolist()
    },
        'offset': data['offset'],
        'initial_estimate': data['initial_estimate'].tolist(),
        'info': {
        'instance': data['instance'],
        'additional_info': data['additional_info'],
        'qubo_multiplication_factor': multiplication
    }
    }

In [ ]:
import os
for d in data_list:
    size = d['instance'][len('kcluster'): d['instance'].find('_')]
    os.makedirs(f'../../../biqbin/k-cluster_with_estimate/{size}', exist_ok=True)
    biq_data = to_biqbin_repr(d)

    # with open(f'../../../biqbin/k-cluster_with_estimate/{size}/{biq_data['info']['instance']}.json', 'w') as f:
    #     json.dump(biq_data, f)